In [1]:
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic
from langsmith import Client
from dotenv import load_dotenv

load_dotenv()


embedding = UpstageEmbeddings(model="solar-embedding-1-large")

database = Chroma(
    embedding_function = embedding,
    collection_name = 'real_estate_tax',
    persist_directory = './real_estate_tax_collection'
)

retriever = database.as_retriever(
    search_type= 'mmr',
    search_kwargs={
        'k': 4,
        'fetch_k': 10,
        'lambda_mult': 0.6
    }
)

client = Client()

# 비용의 80%는 Haiku가, 정확도의 80%는 Sonnet/Opus가!!

smart_llm = ChatAnthropic(
    model='claude-opus-4-5-20251101',
    temperature= 0
)

llm = ChatAnthropic(
    model='claude-sonnet-4-5-20250929',
    temperature= 0
)

small_llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',
    temperature= 0
)

## 도구 선언

In [2]:
from langchain_core.tools import tool

In [3]:
@tool       # 데코레이터 @tool로 감싸고
def add(num1: int, num2: int) -> int:     # 함수와 변수의 이름, typehint가 schema(구조)로서, JSON 형식으로 LLM에 전달됨
    '''
    정수 num1와 num2를 더합니다. 
    '''         # 이 부분의 description(프롬프트와 같음)도 마찬가지로 LLM에 전달됨
    return num1+num2

@tool
def multiply(num1: int, num2: int) -> int:
    '''
    정수 num1와 num2를 곱합니다. 
    '''
    return num1*num2

In [20]:
add(19, 19)     # @tool로 감싸진 tool들은 callable이지 않음

TypeError: 'StructuredTool' object is not callable

In [4]:
# 대신 다른 Runnable처럼 invoke() 하고 변수는 딕셔너리로 감싸져야 함

add.invoke({'num1': 19, 'num2': 19})

38

## LLM에 바인딩 (도구 쥐어주기)

In [5]:
# 원본인 small_llm을 변경하지 않으므로(inplace=False) 새로운 객체에 할당이 요구됨
llm_with_tools = small_llm.bind_tools([add, multiply])

## LLM의 도구 호출

In [6]:
# 그냥 LLM에 호출했을때
response = llm.invoke('19와 19를 더해줘')

In [7]:
from rich import print as rprint

rprint(response.dict())

# 보이듯이 content에 답변이 담겨 돌아온다

{
    'content': '19 + 19 = 38입니다.',
    'additional_kwargs': {},
    'response_metadata': {
        'id': 'msg_015SDvdwjcZayeYhopYAUFb7',
        'model': 'claude-sonnet-4-5-20250929',
        'stop_reason': 'end_turn',
        'stop_sequence': None,
        'usage': {
            'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0},
            'cache_creation_input_tokens': 0,
            'cache_read_input_tokens': 0,
            'input_tokens': 18,
            'output_tokens': 17,
            'server_tool_use': None,
            'service_tier': 'standard'
        },
        'model_name': 'claude-sonnet-4-5-20250929'
    },
    'type': 'ai',
    'name': None,
    'id': 'lc_run--019c2227-ead8-7e21-990d-1c249aa692aa-0',
    'tool_calls': [],
    'invalid_tool_calls': [],
    'usage_metadata': {
        'input_tokens': 18,
        'output_tokens': 17,
        'total_tokens': 35,
        'input_token_details': {'cache_read': 0, 'cache_creation': 0}
    }
}

In [8]:
add_response = llm_with_tools.invoke('19와 19를 더해줘')

In [9]:
from rich import print as rprint

rprint(add_response.dict())

# 답변이 담기는 것이 아닌, LLM의 도구 호출이 돌아온다

{
    'content': [
        {
            'id': 'toolu_01GAYqtEqKcgxBi46astaRRF',
            'input': {'num1': 19, 'num2': 19},
            'name': 'add',
            'type': 'tool_use'
        }
    ],
    'additional_kwargs': {},
    'response_metadata': {
        'id': 'msg_01SNtuSCQLfbNnmcyrfKP4RP',
        'model': 'claude-haiku-4-5-20251001',
        'stop_reason': 'tool_use',
        'stop_sequence': None,
        'usage': {
            'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0},
            'cache_creation_input_tokens': 0,
            'cache_read_input_tokens': 0,
            'input_tokens': 723,
            'output_tokens': 71,
            'server_tool_use': None,
            'service_tier': 'standard'
        },
        'model_name': 'claude-haiku-4-5-20251001'
    },
    'type': 'ai',
    'name': None,
    'id': 'lc_run--019c2227-f69e-7d01-a5f9-df776d5e5ed5-0',
    'tool_calls': [
        {
            'name': 'add',
            'args': {'num1': 19, 'num2': 19},
            'id': 'toolu_01GAYqtEqKcgxBi46astaRRF',
            'type': 'tool_call'
        }
    ],
    'invalid_tool_calls': [],
    'usage_metadata': {
        'input_tokens': 723,
        'output_tokens': 71,
        'total_tokens': 794,
        'input_token_details': {'cache_read': 0, 'cache_creation': 0}
    }
}

In [35]:
message_list = []

In [36]:
from typing import Sequence
from langchain_core.messages import AnyMessage, HumanMessage

query = '19와 19를 더해줘'

human_message = HumanMessage(query)
message_list: Sequence[AnyMessage] = [human_message]

In [37]:
tool_use = llm_with_tools.invoke(message_list)

In [38]:
message_list.append(tool_use)
tool_use.tool_calls

[{'name': 'add',
  'args': {'num1': 19, 'num2': 19},
  'id': 'toolu_01JdZ8pQSsrU7g745185RTpo',
  'type': 'tool_call'}]

In [39]:
tool_message = add.invoke(tool_use.tool_calls[0])
message_list.append(tool_message)

In [40]:
# message_list = [HumanMessage → AIMessage(tool_use) → ToolMessage(tool_result)]
response = llm_with_tools.invoke(message_list)

In [41]:
response

AIMessage(content='19와 19를 더하면 **38**입니다.', additional_kwargs={}, response_metadata={'id': 'msg_0176GtpdPuQa7xWCV7wnqXRV', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 807, 'output_tokens': 20, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001'}, id='lc_run--019c2239-6fba-7763-9b07-b9365dc39e33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 807, 'output_tokens': 20, 'total_tokens': 827, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}})